In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import cudaq
from cudaq import spin
from cudaq.qis import *
import numpy as np
import matplotlib.pyplot as plt
from typing import List

#!pip install scikit-learn
#!pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# GTC 2026 Decoding Quantum Errors with AI

This notebook accompanies the GTC tutorial entitled "Decoding Quantum Errors with AI".  The material below will supplement the talk slides and provides some hands on exercises, interactive widgets, and resources to learn more. You may not complete all of the content below during the workshop, but can use this notebook to reference material later on.

If you are brand new to quantum computing, we suggest you complete the CUDA-Q Academic series "[Quick start to Quantum Computing](https://github.com/NVIDIA/cuda-q-academic/tree/main/quick-start-to-quantum)"

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #76b900; margin-top: 0;">Exercise 1 - The Classical Hamming Code</h3>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
        The widget below runs a demonstration of the classical [7,4,3] Hamming code. Do the following to understand how it works:
    </p>
    <ol style="font-size: 16px; color: #333; line-height: 1.8; margin-left: 16px;">
        <li>Try changing the original message. Notice how changing the four message bits impacts the three parity bits.</li>
        <li>Try adding a single error to each data bit, one at a time. What syndromes are produced for each case?</li>
        <li>Try adding a single error to each parity check bit, one at a time. What syndromes are produced in each case?</li>
        <li>Add an error on any two bits of your choice. What syndrome is produced? What single bit error would this be confused with? What logical error would result?</li>
    </ol>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
        Run the cell below or access the widget in a browser <a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/hamming.html" style="color: #76b900; font-weight: bold;">here</a>.
    </p>
</div>

In [ ]:
from IPython.display import HTML
scale = 0.8  # Change this to zoom: 0.7 = more zoomed out, 1.0 = full size
content_height = 2300  # Full height of the widget content in px
visible_height = int(content_height * scale)

HTML(f'''<div style="overflow:hidden; height:{visible_height}px;">
  <iframe src="https://nvidia.github.io/cuda-q-academic/interactive_widgets/hamming.html"
    style="width:{int(100/scale)}%; height:{content_height}px; border:none;
           transform:scale({scale}); transform-origin:top left;">
  </iframe>
</div>''')

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #76b900; margin-top: 0;">Exercise 2 — The Quantum Repetition Code with CUDA-Q</h3>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    Explore the quantum repetition code implemented below as a CUDA-Q kernel. For background on the derivation, expand the tab below.
    </p>
    <p style="font-size: 16px; color: #333; line-height: 1.5;"><strong>Complete the following:</strong></p>
    <ol style="font-size: 16px; color: #333; line-height: 1.8; margin-left: 20px;">
    <li>Identify the main parts of the kernel: logical encoding of |111⟩, the two parity checks, and the conditional logic that applies corrections from the measured syndromes.</li>
    <li>Run the code (noise-free). What syndrome do you see each run? The logical error rate should be 0.</li>
    <li>Manually add a single bit-flip where indicated using <code>x(data_qubits[i])</code> with <code>i</code> = 0, 1, or 2. Do you see the expected syndrome? Do the corrections remove the error?</li>
    <li>Manually add a second error. What happens to the syndromes and the logical error rate?</li>
    <li>Remove the manual errors and uncomment the code that applies random bit-flip errors. For bit-flip probability <i>p</i> = 0.1, what logical error rate do you get?</li>
    <li>Comment out the syndrome corrections (the <code>x(data_qubits[...])</code> in the <code>if</code> blocks). What happens to the logical error rate? This is the logical error rate without decoding.</li>
    </ol>
    <p style="font-size: 16px; color: #333; line-height: 1.5;"><strong>Optional challenge</strong></p>
    <ul style="font-size: 16px; color: #333; line-height: 1.8; margin-left: 20px;">
    <li>Plot the logical error rate vs. <i>p</i> for 5-, 7-, and 9-qubit repetition codes.</li>
    </ul>
</div>

<details>
<summary><strong>Open this Tab to read optional background on derivation of quantum repetition code</strong></summary>

The section below is an excerpt from "[QEC 101 Lab 1: The Basics of Classical and Quantum Error Correction](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/01_QEC_Intro.ipynb)" and explains the derivation of the quantum repetition code. "[QEC 101 Lab 2 - Stabilizers, the Shor code, and the Steane code](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb)" if you would like to fill in some of these gaps after the tutorial.

Due to the no cloning theorem, the quantum repetition code must be constructed by redundantly encoding a quantum state through quantum entanglement between. Let's start with a generic normalized qubit state $\ket{\psi} = \alpha\ket{0} +\beta\ket{1}$:

The 0 and 1 states can be encoded into a logical state making use of the larger 8-dimensional Hilbert space of three qubits:  

$$\ket{\psi}_L = \alpha\ket{000} +\beta\ket{111} = \alpha\ket{0}_L +\beta\ket{1}_L \ne\ket{\psi} \otimes \ket{\psi} \otimes \ket{\psi} $$ 

Consider now the entire Hilbert space spanned by the eight basis states.  The basis states are separated into the logical codewords that make up the codespace:   

| Codespace    | Notation for the logical codewords| 
| ----------- | ----------- | 
| $\ket{000}$ | $\ket{0}_L$ |
|$\ket{111}$ | $\ket{1}_L$ |

and the remaining basis states make up the error space:

| Error space |
| ----------- | 
| $\ket{001}$ |
| $\ket{010}$ |
| $\ket{100}$ |
| $\ket{011}$ |
| $\ket{101}$ |
| $\ket{110}$ |


The operators $Z_1Z_2$ and $Z_2Z_3$ are stabilizers for the repetition code.  ($Z_n$ returns an eigenvalue of +1 if the nth qubit in a ket is a 0 and -1 if it is a 1).  

In other words, if qubit 1 and 2 are both 0 or both 1, then $Z_1Z_2 \ket{\psi} = \ket{\psi}$. If one of the qubits is 0 and the other 1, then $Z_1Z_2 \ket{\psi} = -\ket{\psi}$. In both cases we obtain the +/- eigenstate and preserve the encoded information.

Using projective measurement, we can extract this eigenvalue using an ancilla qubit for each stabilizer. (The details of the derivation are shown following the code cell below if you are curious). The point is that we can generate a syndrome that checks the parity of the qubits without destroying the state! Each syndrome we can map to a specific single qubit error. 

| $Z_1Z_2$ Syndrome   | $Z_2Z_3$ Syndrome| Data qubit state (after error) | Single Bit Flip Correction 
| ----------- | ----------- | ----------- | ----------- | 
| 0 | 0 | $\ket{111}$ | none |
| 1 | 0 | $\ket{011}$ | $X_1$ |
| 0 | 1 | $\ket{110}$ | $X_3$ |
| 1 | 1 | $\ket{101}$ | $X_2$ |


Like the classical repetition code, there is no way to know for certain if a (1,0) syndrome corresponds to a single bit flip error from the $\ket{111}$ codeword or a two bit flip error from the $\ket{000}$ codeword.  However, it is always prudent to assume that the case with fewer errors is more likely.

Also note, that the repetition code can only catch $X$ or $Z$ error but not both

The entire quantum circuit for the three qubit repetition code is shown below, where the ancilla qubits are used to compute the $Z_1Z_2$ and $Z_2Z_3$ syndromes.

<img src="https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/qec101/Images/repcircuit.png" alt="Three-qubit repetition code circuit" style="width: 800px;"/>

</details>


In [ ]:
import cudaq
try:
    del s0, s1, d0, d1, d2
except NameError:
    pass

# Initialize Noise model
noise_model = cudaq.NoiseModel()

# Set probability of bit-flip error
p = 0.1

@cudaq.kernel
def three_qubit_repetition_code() -> list[bool]:
    data_qubits = cudaq.qvector(3)
    ancilla_qubits = cudaq.qvector(2)

    # Encode logical |1>_L = |111>
    x(data_qubits)

    """
    # Apply bit-flip (X) noise on each data qubit (code-capacity model)
    for i in range(3):
        cudaq.apply_noise(cudaq.XError, p, data_qubits[i])
    """
    
    ## ADD Manual Bitflip Here##
    
    # Extract Syndromes
    h(ancilla_qubits)
    z.ctrl(data_qubits[0], ancilla_qubits[0])
    z.ctrl(data_qubits[1], ancilla_qubits[0])
    z.ctrl(data_qubits[1], ancilla_qubits[1])
    z.ctrl(data_qubits[2], ancilla_qubits[1])
    h(ancilla_qubits)

    s0 = mz(ancilla_qubits[0])
    s1 = mz(ancilla_qubits[1])

    # Correct errors based on syndromes
    if s0 and s1:
        x(data_qubits[1])
    elif s0:
        x(data_qubits[0])
    elif s1:
        x(data_qubits[2])

    # Measure Data bits
    d0 = mz(data_qubits[0])
    d1 = mz(data_qubits[1])
    d2 = mz(data_qubits[2])
    return [s0, s1, d0, d1, d2]




# Run kernel with cudaq.run; each sample is [s0, s1, d0, d1, d2]
samples = cudaq.run(three_qubit_repetition_code, noise_model=noise_model, shots_count=1000)

# Top 10: syndrome (s0, s1) and data bits (d0 d1 d2)
print("Top 10 shots (s0, s1 | data bits):")
for i, samp in enumerate(samples[:20]):
    s0, s1, d0, d1, d2 = samp[0], samp[1], samp[2], samp[3], samp[4]
    data_str = f"{int(d0)}{int(d1)}{int(d2)}"
    print(f"  {i+1}: ({int(s0)}, {int(s1)}) | {data_str}")

# Logical error rate (we encoded |1>_L = |111>; outcome != 111 is a logical error)
total = len(samples)
logical_errors = sum(1 for s in samples if f"{int(s[2])}{int(s[3])}{int(s[4])}" != "111")
logical_error_rate = logical_errors / total if total else 0.0
print(f"\nLogical error rate (fraction of outcomes != 111): {logical_error_rate:.4f} ({logical_errors}/{total})") 

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #76b900; margin-top: 0;">Exercise 3 — MWPM Decoding</h3>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    Use the widget below to get a hands-on feel for how challenging decoding becomes as QEC codes scale. You will decode four scenarios  using a technique called <strong>minimum weight perfect matching (MWPM)</strong>, the leading algorithmic decoder for topological codes such as the toric code and surface code. Understanding the details of these codes is beyond the scope of this tutorial, but we encourage interested learners to try the CUDA-Q Academic lesson <a href="https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/06_QEC_Topological_Codes.ipynb" target="_blank" rel="noopener">Topological Codes</a> to learn more.
    </p>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    You will decode syndromes for the toric code which can be visualized where <strong>edges</strong> are data qubits and <strong>vertices</strong> are ancilla qubits corresponding to stabilizer checks. Red dots signify flagged stabilizers. The optimal decoding result is achieved by pairing all of the red dots so that the overall path between the pairs is minimized. 
    </p>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    there are four scenarions to decoder, the first two with easy and hard syndrome patterns on a smaller distance 7 code. The second two with easy and hard syndrome patterns on a much larger distance 11 code. the main point is that the size of the code and the complexity of the syndrome pattern can determine the difficulty of the decoding task.
    </p>
</div>

In [ ]:
from IPython.display import HTML
import base64

with open("mwpm-decoding-challenge.html", "rb") as f:
    b64 = base64.b64encode(f.read()).decode("ascii")

HTML(f'<iframe src="data:text/html;base64,{b64}" style="width:100%; height:1800px; border:none;"></iframe>')

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #76b900; margin-top: 0;">Exercise 4 - Decoder Metrics</h3>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    Use the decoder simulator function to calculate the number of undecoded syndromes as a function of wall clock time for various settings. Note, this is a qualitative demonstration. An accurate preduction would require much more nuance and system specific information. 
    </p>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    The cells below use <code>QECSimulator</code> to visualize decoder throughput. It takes:
    </p>
    <ul style="font-size: 16px; color: #333; line-height: 1.8; margin-left: 20px;">
    <li><code>syndrome_time_us</code> — time for one syndrome extraction round on the QPU (μs)</li>
    <li><code>min_batch_size</code> — minimum syndromes before a batch is sent to the decoder (sends everything in the backlog)</li>
    <li><code>decode_fun</code> — lambda that gives time to decode <code>n</code> syndromes</li>
    <li><code>max_batches_to_run</code> — maximum number of decoding rounds</li>
    <li><code>n_processors</code> — number of processors for parallel-window decoding (keep 1 for now)</li>
    </ul>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    The output shows total system load (syndrome backlog and syndromes being decoded) vs wall-clock time with respect to the QPU, plus a bar chart of decoder time per round. We will run the following scenarios in the talk; try the inputs below alongside.
    </p>
    <p style="font-size: 16px; color: #333; line-height: 1.5;"><strong>Scenarios to run</strong></p>
    <ol style="font-size: 16px; color: #333; line-height: 1.8; margin-left: 20px;">
    <li>Run the default settings. Is the decoder throughput sufficient?</li>
    <li>Keep the decode function linear, but with too little throughput (<code>decode_func=lambda n: n * 1.2</code>). What happens?</li>
    <li>Try a decode function that scales quadratically with a small prefactor (<code>decode_func=lambda n: 0.001 * n**2</code>). Can poorly scaling decoders still work with small problem sizes?</li>
    <li>Try a decode function that scales quadratically with a large prefactor (<code>decode_func=lambda n: n**2</code>). How many rounds complete?</li>
    <li>Keep this decoder function, but try longer syndrome extraction time (<code>syndrome_time_us=200</code>), a reasonable time for an ion trap. What happens?</li>
    </ol>
</div>

In [ ]:
from decoder_simulator import QECSimulator

simulation = QECSimulator(
    syndrome_time_us=1.0,           # Time (microseconds) it takes to perform a syndrome extraction round
    min_batch_size=20,              # minimum number of syndromes required to send queue to decoder
    decode_func=lambda n: n * 1.2,  # function to compute the time (microseconds) it takes to decode n syndromes.
    max_batches_to_run=15,          # maximum number of decoder batches run to halt simulation
    n_processors=1                  # number of processors used for temporal paralell window decoding.
)
simulation.run()
simulation.plot_results("Simulation Results")

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px; border-radius: 4px; font-family: sans-serif;">
    <h3 style="color: #76b900; margin-top: 0;">Exercise 5 - Decoding the Steane Code with AI</h3>
    <p style="font-size: 16px; color: #333; line-height: 1.5;">
    Work through the code cells below to prepare the data for, train, and utilize and basic AI decoding model for the Steane code. After you have run all of the code sections, we will discuss some of the key takeaways.  
    </p>
</div>

The first step is to generate data for training an AI decoder. We will use the Steane code [[7,1,3]], which is the quantum version of the Hamming code. It is a CSS code so it can correct $X$ and $Z$ errors independently, but we will only consider bitflip ($X$) errors for simplicity. You can learn about the Steane code in detail in [QEC 101 Lab 2 — Stabilizers, the Shor code, and the Steane code](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb).

We consider a simple code capacity noise model, where errors only occur between logical encoding and the first syndrome extraction but not at measurement or between syndrome extraction rounds.  When training an AI decoder with experimental data, no noise model is required. The decoder just learns the noise that comes from the QPU.

The explicit way to generate this data is by writing a CUDA-Q kernel for the Steane code, applying bitflip errors (or other errors if we choose), and using `cudaq.run()` to sample measurements of the data qubits and ancilla qubits.

The ancilla qubits produce model input data (syndromes) while the total parity of our measured data qubits provides the true logical state labels.

**Note:** The Steane code is a CSS code, so an analogous circuit can be run to catch Z errors. For simplicity, we only consider X errors, but the workflow is nearly identical.

In [ ]:
import cudaq
import numpy as np

# Defines noise model and probability of bit-flip error.
p = 0.05
cudaq.unset_noise()
noise = cudaq.NoiseModel()

@cudaq.kernel
def steane_code() -> list[int]:
    """Runs the Steane code and returns measurement outcomes.
    Returns
    -------
    list[int]
        Measurement outcomes (data qubits then ancilla qubits).
    """   
    data_qubits = cudaq.qvector(7)
    ancilla_qubits = cudaq.qvector(3)

    # Create a superposition over all possible combinations of parity check bits
    h(data_qubits[4])
    h(data_qubits[5])
    h(data_qubits[6])

    #Prepares logical 0 state
    x.ctrl(data_qubits[0],data_qubits[1])
    x.ctrl(data_qubits[0],data_qubits[2])

    x.ctrl(data_qubits[4],data_qubits[0])
    x.ctrl(data_qubits[4],data_qubits[1])
    x.ctrl(data_qubits[4],data_qubits[3])

    x.ctrl(data_qubits[5],data_qubits[0])
    x.ctrl(data_qubits[5],data_qubits[2])
    x.ctrl(data_qubits[5],data_qubits[3])

    x.ctrl(data_qubits[6],data_qubits[1])
    x.ctrl(data_qubits[6],data_qubits[2])
    x.ctrl(data_qubits[6],data_qubits[3])
    
    # Applies Noise here
    for j in range(7):
        cudaq.apply_noise(cudaq.XError, p, data_qubits[j])

    # Syndrome extraction to flag X errors
    h(ancilla_qubits)

    z.ctrl(ancilla_qubits[0],data_qubits[0])
    z.ctrl(ancilla_qubits[0],data_qubits[1])
    z.ctrl(ancilla_qubits[0],data_qubits[3])
    z.ctrl(ancilla_qubits[0],data_qubits[4])

    z.ctrl(ancilla_qubits[1],data_qubits[0])
    z.ctrl(ancilla_qubits[1],data_qubits[2])
    z.ctrl(ancilla_qubits[1],data_qubits[3])
    z.ctrl(ancilla_qubits[1],data_qubits[5])

    z.ctrl(ancilla_qubits[2],data_qubits[1])
    z.ctrl(ancilla_qubits[2],data_qubits[2])
    z.ctrl(ancilla_qubits[2],data_qubits[3])
    z.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)
    
    #Measure all qubits
    d0=mz(data_qubits[0])
    d1=mz(data_qubits[1])
    d2=mz(data_qubits[2])
    d3=mz(data_qubits[3])
    d4=mz(data_qubits[4])
    d5=mz(data_qubits[5])
    d6=mz(data_qubits[6])
    a0=mz(ancilla_qubits[0])
    a1=mz(ancilla_qubits[1])
    a2=mz(ancilla_qubits[2])
    
    #return measurements so we can use cudaq.run().
    #Requires return type when kernel defined.
    return [d0,d1,d2,d3,d4,d5,d6,a0,a1,a2]


The previous cell is quite a bit of work, and requires you to manually construct the entire QEC code. This is not ideal when you are primarily interested in testing an AI decoder and want data generation streamlined. 

Another more efficient way to sample training data for memory experiments is directly using the parity check matrix. Random bitflips can be applied and syndromes determined via matrix multiplication.  CUDA-Q QEC can do this with just a few lines of codes (shown below) and generate the same sort of data we did above with a preloaded Steane code. Additionally, if you want to generate data for multiple syndrome extraction rounds, you can use the `sample_memory_circuit`.  If you want to test a new, non-standard code, you would need to define the kernels explicitly similar to the example above as shown in the docs [here](https://nvidia.github.io/cudaqx/components/qec/introduction.html#core-components)

In [ ]:
import cudaq_qec as qec

steane = qec.get_code("steane")

Hz = steane.get_parity_z()

observable = steane.get_observables_z()

data = qec.generate_random_bit_flips(Hz.shape[1], p)

syndrome = Hz @ data % 2
print(f"syndrome: {syndrome}")

actual_observable = observable @ data % 2
print(f"actual_observable: {actual_observable}")


Returning to our explicit kernel above, we can generate data with `cudaq.run`. This command executes the kernel `nsamples` times and stores the measurements in `samples`.  We can extract the data to get pairs of syndromes from the ancilla nmeasurements and use the data qubits to compute the true logical state. 

The first 10 syndromes and logical state labels are printed.  We can also calculate the raw logical error rate, which is the number of instances where a logical flip occurred from 0 to 1 without any decoding.  Our goal is to lower this using the AI decoder.

In [ ]:
# Generate Data
nsamples = 5000
syndromes = []
logical_flips =[]

samples = cudaq.run(steane_code, noise_model=noise, shots_count=nsamples)

for samp in samples:
    ancilla = [samp[7], samp[8], samp[9]]
    data = [samp[0], samp[1], samp[2],samp[3],samp[4],samp[5],samp[6]]
    syndromes.append(ancilla)
    parity = sum(data) %2
    logical_flips.append(parity)

raw_logical_error_rate = sum(logical_flips)/len(logical_flips)
print("Raw Logical Error Rate:", raw_logical_error_rate)

print(syndromes[0:10])
print(logical_flips[0:10])

We are now ready to train a simple multi-layer perceptron model using the data.  The model below has three input neurons (one for each ancilla) followed by two hidden layers of 32 and 16 neurons, and ends with a single output neuron corresponding to the predicted (or decoded) logical state.

The cell below splits the data into test and training sets for validation and then trains the model using PyTorch. 

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# This section normalizes and loads the data you defined previously
syndromes = np.array(syndromes, dtype=np.float32)
logical_flips = np.array(logical_flips, dtype=np.float32)

# Normalize input data
syndromes = (syndromes - syndromes.mean()) / syndromes.std()

X_train, X_test, y_train, y_test = train_test_split(
    syndromes, logical_flips, test_size=0.20, random_state=42)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

# Create data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

# This builds a simple NN model which will be trained
class SimpleDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.act = nn.Sigmoid()
        
        # Initialize weights
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.act(self.fc3(x))

model = SimpleDecoder()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=10,)

# Define the test_accuracy function
def test_accuracy():
    model.eval()
    with torch.no_grad():
        pred = (model(X_test).squeeze() > 0.5).float()
        return (pred == y_test).float().mean().item()

# Training loop
num_epochs = 30
train_losses, test_acc = [], []

# epoch 0 (untrained)
test_acc.append(test_accuracy())

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0
    batch_count = 0
    
    # Training
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        out = model(batch_X).squeeze()
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    # Calculate average loss for the epoch
    avg_epoch_loss = epoch_loss / batch_count
    train_losses.append(avg_epoch_loss)
    
    # Evaluate accuracy
    current_acc = test_accuracy()
    test_acc.append(current_acc)
    
    # Update learning rate
    scheduler.step(avg_epoch_loss)
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:3d}/{num_epochs} | "
              f"train loss={avg_epoch_loss:.4f} | "
              f"test acc={current_acc:.4f}")

# Plotting
plt.figure(figsize=(12,4))

plt.subplot(1, 2, 1)
plt.plot(range(num_epochs + 1), test_acc, marker='o')
plt.xlabel("Training Epoch")
plt.ylabel("Test Accuracy")
plt.title("Steane-Code Decoder Accuracy")
plt.grid(True)
plt.ylim(0, 1)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_losses, color='red')
plt.xlabel("Training Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Time")
plt.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Results:")
print(f"Final Test Accuracy: {test_acc[-1]:.4f}")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Raw (Undecoded) Accuracy: {1-raw_logical_error_rate:.4f}")

# Print first 20 predictions
model.eval()
with torch.no_grad():
    test_predictions = model(X_test).squeeze()
    predicted_labels = (test_predictions > 0.5).float()

print("\nFirst 20 test examples:")
print("=" * 40)
print(f"{'Index':^6} | {'True':^6} | {'Predicted':^6}")
print("=" * 40)

for i in range(20):
    true_label = int(y_test[i].item())
    pred_label = int(predicted_labels[i].item())
    print(f"{i:^6} | {true_label:^6} | {pred_label:^6}")

print("=" * 40)

# Calculate accuracy for these 20 examples
correct = (predicted_labels[:20] == y_test[:20]).sum().item()
print(f"\nAccuracy for these 20 examples: {correct}/20 = {correct/20:.2%}")

You should see the model successfully train!  The test set accuracy should increase while the loss functions decreases. 

There are a few key observations to discuss. 

1. If the model parameters are random and we run this training multiple times, we should see the model on average start with an accuracy of about 0.5. This means we would have as much luck flipping a coin as our decoder.  It may start higher or lower depending on the initial parameters bias towards outputting 1's or 0's. So, we do demonstrate the model did train.
   
2. The trained model does outperform the raw logical error rate without decoding. So, our AI decoder is an improvement. Given the simplicity of the Steane code, this is unsurprising, as there is not really any hidden insight to be gleaned as we can essentially work out the brute force MLE decoding by hand.

3. The final output of the AI model (or any decoder) is limited by the underlying QEC code and its distance.  This is what determines what errors are detectable or correctable before we even try decoding. For example, the distance three Steane code cannot correct two errors. So, we cannot expect the AI model to learn how to correct these errors either. Thus, AI decoding shines in the regime where there are many correctable errors with non-trivial syndrome patterns.

4. Because the trained decoder depends on the error model used, it is really important to have large training data sets with sufficiently realistic noise models and model often require fine tuning with experimental data to realize peak performance. If the error rate was tiny, the model may just learn to output logical 0 all the time, and learn nothing about the error patterns if it has insufficient cases to train on. When training on physical QPU data, we are not trying to learn a noise model, but the actual, unknown noise profile of the device.


## Additional Resources

We have just scratched the surface of AI decoding, and there are many more advanced topics that can be covered from here. Below is a list of some specific resources in the CUDA-Q QEC docs or other GTC material that covers other aspects of AI decoding or GTC more generally. If you want to learn more about QEC, check out our CUDA-Q Academic course entitled QEC 101.



Tools:
CUDA-Q Docs
CUDA-Q QEC Docs


Hands on Learning:
CUDA-Q Academic (All Courses)
CUDA-Q Academic (QEC 101)
CUDA-Q Applications Hub
CUDA-Q QEC Examples
Widget Gallery